# Project 2: Sequence-to-Sequence & Alternative Sequence Generation

This project implements an **Encoder-Decoder Sequence-to-Sequence (Seq2Seq)** model using GRU/LSTM to generate **alternative text sequences** (e.g. Paraphrasing / Machine Translation / Text Re-writing) trained on real sequence pairs.

---

## Key Topics Covered:

1. **Sequence-to-Sequence Architecture**: Encoder GRU encoding variable-length source sequences into a context vector $\mathbf{h}_T$, and Decoder GRU generating target sequences.
2. **Teacher Forcing Strategy**: Passing true target tokens vs predicted tokens during training to accelerate convergence.
3. **Real Sequence Pair Data Pipeline**: Data processing, padding, special tokens (`<SOS>`, `<EOS>`, `<PAD>`, `<UNK>`).
4. **Autoregressive Sequence Decoding**: Generating alternative sequence outputs step-by-step until `<EOS>` token is reached.


In [1]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}, PyTorch Version: {torch.__version__}")


Device: cpu, PyTorch Version: 2.11.0+cpu


## 1. Real Parallel Dataset & Special Tokens Setup

We build special tokens (`<PAD>=0`, `<SOS>=1`, `<EOS>=2`, `<UNK>=3`) essential for variable length sequence processing.


In [2]:
PAD_TOKEN = 0
SOS_TOKEN = 1
EOS_TOKEN = 2
UNK_TOKEN = 3

class Vocabulary:
    def __init__(self):
        self.word2idx = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.idx2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.num_words = 4

    def add_sentence(self, sentence):
        for word in sentence.lower().split():
            if word not in self.word2idx:
                self.word2idx[word] = self.num_words
                self.idx2word[self.num_words] = word
                self.num_words += 1

# Real sequence pairs (Source sequence -> Alternative target sequence)
real_sequence_pairs = [
    ("how are you doing today", "how is your day going"),
    ("what is your name", "who are you"),
    ("where are you going", "what is your destination"),
    ("i am feeling happy", "i feel joyful"),
    ("could you please help me", "can you give me a hand"),
    ("the weather is very nice", "it is a pleasant day outside"),
    ("i love deep learning", "i enjoy neural networks"),
    ("see you later my friend", "catch you afterwards buddy")
]

src_vocab = Vocabulary()
tgt_vocab = Vocabulary()

for src, tgt in real_sequence_pairs:
    src_vocab.add_sentence(src)
    tgt_vocab.add_sentence(tgt)

print(f"Source Vocab Size: {src_vocab.num_words}")
print(f"Target Vocab Size: {tgt_vocab.num_words}")


Source Vocab Size: 34
Target Vocab Size: 31


## 2. Encoder & Decoder GRU Modules


In [3]:
class EncoderGRU(nn.Module):
    def __init__(self, input_dim, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embed_dim, padding_idx=PAD_TOKEN)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        
    def forward(self, src):
        # src: (B, Src_Len)
        embeds = self.embedding(src) # (B, Src_Len, Embed_Dim)
        outputs, hidden = self.gru(embeds) # hidden: (1, B, Hidden_Dim)
        return outputs, hidden

class DecoderGRU(nn.Module):
    def __init__(self, output_dim, embed_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, embed_dim, padding_idx=PAD_TOKEN)
        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, input_step, hidden):
        # input_step: (B, 1)
        embeds = self.embedding(input_step) # (B, 1, Embed_Dim)
        output, hidden = self.gru(embeds, hidden) # (B, 1, Hidden_Dim)
        prediction = self.fc_out(output.squeeze(1)) # (B, Output_Dim)
        return prediction, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        
    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        # src: (B, Src_Len), tgt: (B, Tgt_Len)
        batch_size = src.size(0)
        tgt_len = tgt.size(1)
        tgt_vocab_size = self.decoder.fc_out.out_features
        
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size, device=src.device)
        
        _, hidden = self.encoder(src)
        
        # First input to decoder is <SOS> token
        decoder_input = tgt[:, 0].unsqueeze(1)
        
        for t in range(1, tgt_len):
            prediction, hidden = self.decoder(decoder_input, hidden)
            outputs[:, t, :] = prediction
            
            # Teacher forcing: decide whether to feed actual target or predicted target
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = prediction.argmax(1).unsqueeze(1)
            decoder_input = tgt[:, t].unsqueeze(1) if teacher_force else top1
            
        return outputs

encoder = EncoderGRU(src_vocab.num_words, embed_dim=64, hidden_dim=128)
decoder = DecoderGRU(tgt_vocab.num_words, embed_dim=64, hidden_dim=128)
seq2seq_model = Seq2Seq(encoder, decoder).to(device)

print(seq2seq_model)


Seq2Seq(
  (encoder): EncoderGRU(
    (embedding): Embedding(34, 64, padding_idx=0)
    (gru): GRU(64, 128, batch_first=True)
  )
  (decoder): DecoderGRU(
    (embedding): Embedding(31, 64, padding_idx=0)
    (gru): GRU(64, 128, batch_first=True)
    (fc_out): Linear(in_features=128, out_features=31, bias=True)
  )
)


## 3. Alternative Sequence Inference & Generation Function


In [4]:
def generate_alternative_sequence(model, src_sentence, max_len=15):
    model.eval()
    model.to("cpu")
    
    tokens = src_sentence.lower().split()
    src_ids = [src_vocab.word2idx.get(w, UNK_TOKEN) for w in tokens]
    src_tensor = torch.tensor([src_ids], dtype=torch.long)
    
    with torch.no_grad():
        _, hidden = model.encoder(src_tensor)
        
        decoder_input = torch.tensor([[SOS_TOKEN]], dtype=torch.long)
        result_indices = []
        
        for _ in range(max_len):
            prediction, hidden = model.decoder(decoder_input, hidden)
            top1 = prediction.argmax(1).item()
            
            if top1 == EOS_TOKEN:
                break
                
            result_indices.append(top1)
            decoder_input = torch.tensor([[top1]], dtype=torch.long)
            
    alternative_words = [tgt_vocab.idx2word.get(idx, "<UNK>") for idx in result_indices]
    return " ".join(alternative_words)

# Demo run
test_src = "how are you doing today"
print("Input Source Sequence:    ", test_src)
print("Generated Alternative Seq:", generate_alternative_sequence(seq2seq_model, test_src))


Input Source Sequence:     how are you doing today
Generated Alternative Seq: give how
